# DSE/QF4212 Project

Installing requirements and Importing required files

In [ ]:
# Requirements
# %pip install pandas numpy matplotlib fredapi python-dotenv scipy scikit-learn joblib tdqm xgboost seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

from fredapi import Fred

import os
from dotenv import load_dotenv
from scipy.optimize import minimize
from scipy import stats
from scipy.stats import spearmanr

from sklearn.covariance import LedoitWolf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

import pickle
import glob
from pathlib import Path
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


from sklearn.metrics import r2_score

from joblib import Parallel, delayed
from tqdm.auto import tqdm

from itertools import product
from multiprocessing import Process

# from src import * Importing our own modules

API_KEY='084a38e9d6fd82146cf89b8c30eba224'
random.seed(4212)

## Preparing Data for next Step

In [ ]:
df = pd.read_csv('all_stocks_5yr.csv', index_col=0, parse_dates=True)

close = df.pivot_table(index='date', columns='Name', values='close').dropna(axis=1)
op = df.pivot_table(index='date', columns='Name', values='open')[[col for col in close.columns]]
high = df.pivot_table(index='date', columns='Name', values='high')[[col for col in close.columns]]
low = df.pivot_table(index='date', columns='Name', values='low')[[col for col in close.columns]]
volume = df.pivot_table(index='date', columns='Name', values='volume')[[col for col in close.columns]]

op.fillna(method='ffill', inplace=True)
high.fillna(method='ffill', inplace=True)
low.fillna(method='ffill', inplace=True)
volume.replace(0, np.nan, inplace=True)
volume.fillna(method='ffill', inplace=True)

returns = np.log(close).diff().dropna()

fred = Fred(api_key = API_KEY)
risk_free_rate = fred.get_series_latest_release('DGS3MO')/100/252
risk_free_rate = risk_free_rate.reindex(returns.index, method='ffill')
risk_free_rate.fillna(method='ffill', inplace=True)


with open('processed_data.pkl', 'wb') as f:
    pickle.dump((op, high, low, close, volume, returns, risk_free_rate), f)

## Feature Engineering

In [ ]:
class FeatureEngineer:
    """
    Compute time-series and cross-sectional features from OHLC data
    """
    
    def __init__(self, 
                 open: pd.DataFrame,
                 high: pd.DataFrame,
                 low: pd.DataFrame,
                 close: pd.DataFrame,
                 volume: pd.DataFrame,
                 returns: pd.DataFrame,
                 risk_free_rate,
                 lookback_days: int = 60):
        """
        Args:
            ohlc_df: Long format DataFrame with columns: 
                     ['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']
            rf_rate: Series with date index and annualized 3-month T-bill rate (%)
            lookback_days: Minimum days of history needed for features
        """
        print("Initializing Feature Engineer...")

        self.close = close
        self.open = open
        self.high = high
        self.low = low
        self.volume = volume
        self.rf_rate = risk_free_rate 

        # Calculate returns
        self.returns = returns
        
        # Excess returns
        self.excess_returns = self.returns.sub(self.rf_rate, axis=0)
        
        self.lookback = lookback_days
        self.tickers = self.close.columns.tolist()
        self.dates = self.returns.index
        
        print(f"  Stocks: {len(self.tickers)}")
        print(f"  Date range: {self.dates[0].date()} to {self.dates[-1].date()}")
        print(f"  Lookback period: {lookback_days} days")

    def _compute_features_and_targets_generic(self,rebalance_dates, horizon_days: int, target: str, save_path: str = None) -> pd.DataFrame:
        """
        Generic method for any frequency/horizon combination
        
        Args:
            rebalance_dates: List of dates to compute features for
            horizon_days: Prediction horizon in days
            target: Target variable to compute ('return', 'volatility')
            save_path: Optional path to save features as pickle
        
        Returns:
            all_features: DataFrame with computed features
            targets_df: DataFrame with target variable
        """
        print("\nComputing features...")
        
        # Only compute for dates with sufficient history
        start_date = self.dates[self.lookback]
        valid_dates = [d for d in rebalance_dates if d >= start_date]
        
        all_features = {}
        targets_dict = {}

        print(f"  Total dates to compute: {len(valid_dates)}")
        
        for i, date in enumerate(valid_dates):
            if i % 50 == 0 or i == len(valid_dates) - 1:
                print(f"  Progress: {i+1}/{len(valid_dates)} ({(i+1)/len(valid_dates)*100:.1f}%)")
            
            # Get data up to (but not including) this date
            date_idx = self.dates.get_loc(date)
            window_start = date_idx - self.lookback
            window_end = date_idx
            
            # Extract windows
            returns_window = self.returns.iloc[window_start:window_end]
            excess_returns_window = self.excess_returns.iloc[window_start:window_end]
            close_window = self.close.iloc[window_start:window_end]
            volume_window = self.volume.iloc[window_start:window_end]
            high_window = self.high.iloc[window_start:window_end]
            low_window = self.low.iloc[window_start:window_end]

            # Compute features for this date
            date_features = self._compute_features_single_date(
                returns_window, 
                excess_returns_window,
                close_window,
                volume_window,
                high_window,
                low_window
            )         

            date_features = self.feature_subset_horizon(date_features, horizon_days, target)   

            date_features = date_features.set_index('ticker')
            all_features[date] = date_features
            if save_path:
                if not date_features.isna().values.any():
                    with open(save_path+str(date.date())+".pkl", 'wb') as f:
                        pickle.dump(date_features, f)  
                else:
                    print(date)   
        
            # Compute target
            if target == 'return':
                future_returns = self.returns.iloc[date_idx+1 : date_idx+1+horizon_days]
                targets = future_returns.sum(axis=0)  # Sum across days
            else:
                future_returns = self.returns.iloc[date_idx+1 : date_idx+1+horizon_days]
                rv = (future_returns ** 2).sum(axis=0)
                targets = rv * (252 / horizon_days)  # Annualise 
            targets_dict[date] = targets
        
        targets_dict = pd.DataFrame(targets_dict).T
        targets_dict.sort_index(inplace=True)

        if save_path:
            if not targets_dict.isna().values.any():
                with open(f"{save_path}{target}_{horizon_days}_targets.pkl", 'wb') as f:
                    pickle.dump(targets_dict, f)  
            else:
                print(date) 


        # Combine all dates
        features_df = pd.concat(all_features, ignore_index=True)
        
        print(f"\n✓ Feature computation complete!")
        print(f"  Shape: {features_df.shape}")
        print(f"  Features: {features_df.shape[1]}")
        print(f"  Memory: {features_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

        
        return features_df, targets_dict
    
    def compute_daily_return_features(self, save_path=None):
        dates = self.dates[self.lookback:]  # All dates
        return self._compute_features_and_targets_generic(
            dates, horizon_days=1, target='return', save_path=save_path
        )
    
    def compute_weekly_return_features(self, save_path=None):
        dates = self._get_weekly_dates()
        return self._compute_features_and_targets_generic(
            dates, horizon_days=5, target='return', save_path=save_path
        )
    
    def compute_monthly_return_features(self, save_path=None):
        dates = self._get_monthly_dates()
        return self._compute_features_and_targets_generic(
            dates, horizon_days=21, target='return', save_path=save_path
        )
    
    def compute_weekly_volatility_features(self, save_path=None):
        dates = self._get_weekly_dates()
        return self._compute_features_and_targets_generic(
            dates, horizon_days=5, target='volatility', save_path=save_path
        )
    
    def compute_monthly_volatility_features(self, save_path=None):
        dates = self._get_monthly_dates()
        return self._compute_features_and_targets_generic(
            dates, horizon_days=21, target='volatility', save_path=save_path
        )

    def _get_weekly_dates(self):
        """
        Get weekly rebalance dates (Mondays or first trading day of week)
        
        Returns:
            List of weekly dates
        """
        # Resample to weekly (Monday)
        weekly_dates = self.returns.groupby([self.returns.index.year, self.returns.index.isocalendar().week]).apply(lambda x: x.index[0]).tolist()
        # Ensure we have enough lookback history
        min_date = self.dates[self.lookback]
        weekly_dates = [d for d in weekly_dates if d >= min_date]
        
        return weekly_dates
    
    def _get_monthly_dates(self):
        """First trading day of each month"""
        monthly_dates = self.returns.groupby([self.returns.index.year, self.returns.index.month]).apply(lambda x: x.index[0]).tolist()
        min_date = self.dates[self.lookback]
        return [d for d in monthly_dates if d >= min_date]
    
    def _compute_features_single_date(self,
                                    returns_window: pd.DataFrame,
                                    excess_returns_window: pd.DataFrame,
                                    close_window: pd.DataFrame,
                                    volume_window: pd.DataFrame,
                                    high_window: pd.DataFrame,
                                    low_window: pd.DataFrame) -> pd.DataFrame:
        """
        Compute ALL features for one date across all stocks
        
        No for_return flag - computes complete feature set
        Feature selection happens in feature_subset_horizon()
        
        Returns:
            DataFrame with ~60-70 features per stock
        """
        n_stocks = len(self.tickers)
        features_list = []
        
        # === TIME-SERIES FEATURES (per stock) ===
        for i, ticker in enumerate(self.tickers):
            feat = {'ticker': ticker}
            
            ret = returns_window.iloc[:, i].values
            excess_ret = excess_returns_window.iloc[:, i].values
            price = close_window.iloc[:, i].values
            vol = volume_window.iloc[:, i].values
            high = high_window.iloc[:, i].values
            low = low_window.iloc[:, i].values
            

            # MOMENTUM FEATURES

            # Short-term momentum
            feat['ret_1d'] = ret[-1] if len(ret) > 0 else 0
            feat['ret_5d'] = np.sum(ret[-5:]) if len(ret) >= 5 else 0
            
            # Medium-term momentum
            feat['ret_21d'] = np.sum(ret[-21:]) if len(ret) >= 21 else 0
            feat['excess_ret_21d'] = np.sum(excess_ret[-21:]) if len(excess_ret) >= 21 else 0
            
            # Long-term momentum
            feat['ret_63d'] = np.sum(ret[-63:]) if len(ret) >= 63 else 0
            feat['excess_ret_63d'] = np.sum(excess_ret[-63:]) if len(excess_ret) >= 63 else 0
            
            # VOLATILITY FEATURES
            
            # Standard deviation measures
            feat['vol_5d'] = np.std(ret[-5:]) if len(ret) >= 5 else 0
            feat['vol_21d'] = np.std(ret[-21:]) if len(ret) >= 21 else 0
            feat['vol_63d'] = np.std(ret[-63:]) if len(ret) >= 63 else 0
            
            # Realised variance (mean of squared returns)
            feat['rv_5d'] = np.mean(ret[-5:]**2) if len(ret) >= 5 else 0
            feat['rv_21d'] = np.mean(ret[-21:]**2) if len(ret) >= 21 else 0
            feat['rv_63d'] = np.mean(ret[-63:]**2) if len(ret) >= 63 else 0
            
            # EWMA volatility
            if len(ret) >= 21:
                lambda_decay = 0.94
                ewma_var = 0
                for j in range(len(ret)-1, max(len(ret)-22, 0), -1):
                    ewma_var = lambda_decay * ewma_var + (1 - lambda_decay) * ret[j]**2
                feat['vol_ewma'] = np.sqrt(ewma_var) if ewma_var > 0 else 0
            else:
                feat['vol_ewma'] = 0
            
            # Downside volatility (semi-deviation)
            if len(ret) >= 21:
                downside_returns = ret[-21:][ret[-21:] < 0]
                feat['downside_vol_21d'] = np.std(downside_returns) if len(downside_returns) > 0 else 0
            else:
                feat['downside_vol_21d'] = 0
            
            # ==========================================
            # VOLATILITY DYNAMICS
            # ==========================================
            
            # Realised variance ratios
            feat['rv_ratio_5_21'] = feat['rv_5d'] / feat['rv_21d'] if feat['rv_21d'] > 0 else 0
            feat['rv_ratio_5_63'] = feat['rv_5d'] / feat['rv_63d'] if feat['rv_63d'] > 0 else 0
            feat['rv_ratio_21_63'] = feat['rv_21d'] / feat['rv_63d'] if feat['rv_63d'] > 0 else 0
            
            # Realised variance trend
            feat['rv_trend'] = (feat['rv_21d'] - feat['rv_63d']) / feat['rv_63d'] if feat['rv_63d'] > 0 else 0
            
            # ==========================================
            # TECHNICAL INDICATORS
            # ==========================================
            
            # Moving average ratios
            if len(price) >= 21:
                ma20 = np.mean(price[-21:])
                feat['price_ma20'] = (price[-1] / ma20) - 1
            else:
                feat['price_ma20'] = 0
            
            if len(price) >= 50:
                ma50 = np.mean(price[-50:])
                feat['price_ma50'] = (price[-1] / ma50) - 1
            else:
                feat['price_ma50'] = 0
            
            # Momentum acceleration
            if len(ret) >= 21:
                recent = np.sum(ret[-5:])
                older = np.sum(ret[-21:-5]) / 16 * 5  # Normalise to same period
                feat['momentum_accel'] = recent - older
            else:
                feat['momentum_accel'] = 0
            
            # High-low range position
            if len(high) >= 21:
                high_21 = np.max(high[-21:])
                low_21 = np.min(low[-21:])
                if high_21 != low_21:
                    feat['range_position'] = (price[-1] - low_21) / (high_21 - low_21)
                else:
                    feat['range_position'] = 0.5
            else:
                feat['range_position'] = 0.5
            
            # RSI (14-period)
            if len(price) >= 15:
                deltas = np.diff(price[-15:])
                ups = deltas[deltas > 0].sum() / 14
                downs = -deltas[deltas < 0].sum() / 14
                rs = ups / downs if downs > 0 else 0
                feat['rsi_14'] = 100 - (100 / (1 + rs))
            else:
                feat['rsi_14'] = 50
            
            # Drawdown from peak
            if len(price) >= 21:
                peak = np.max(price[-21:])
                feat['drawdown_21d'] = (price[-1] / peak) - 1
            else:
                feat['drawdown_21d'] = 0
            
            # ==========================================
            # LAGGED RETURNS (for daily predictions)
            # ==========================================
            
            for lag in range(1, 21):
                feat[f'ret_lag_{lag}d'] = ret[-lag] if len(ret) >= lag else 0

            # LAGGED RETURNS (for weekly predictions)
            for lag in range(1, 5):
                feat[f'ret_lag_{(lag-1)*5+1}-{lag*5+1}w'] = ret[-lag * 5-1:-(lag-1)*5-1].sum() if len(ret) >= lag * 5 else 0

            # LAGGED RETURNS (for monthly predictions)
            for lag in range(1, 3):
                feat[f'ret_lag_{(lag-1)*21+1}-{lag*21+1}w'] = ret[-lag * 21-1:-(lag-1)*21-1].sum() if len(ret) >= lag * 21 else 0

            # ==========================================
            # VOLUME FEATURES
            # ==========================================
            
            # Volume ratio
            if len(vol) >= 21:
                avg_vol = np.mean(vol[-21:])
                current_vol = vol[-1]
                feat['volume_ratio'] = current_vol / avg_vol if avg_vol > 0 else 1
            else:
                feat['volume_ratio'] = 1
            
            # Volume volatility
            if len(vol) >= 21:
                feat['volume_vol_21d'] = np.std(vol[-21:]) / (np.mean(vol[-21:]) + 1e-8)
            else:
                feat['volume_vol_21d'] = 0
            
            # ==========================================
            # LIQUIDITY FEATURES
            # ==========================================
            
            if len(ret) >= 21 and len(vol) >= 21:
                # Amihud illiquidity measure
                abs_returns_sum = np.abs(ret[-21:]).sum()
                volume_sum = vol[-21:].sum()
                feat['illiquidity_21d'] = abs_returns_sum / (volume_sum + 1e-8)
                
                # High-low spread proxy
                hl_spreads = (high[-21:] - low[-21:]) / (close_window.iloc[:, i].values[-21:] + 1e-8)
                feat['hl_spread_21d'] = np.mean(hl_spreads)
            else:
                feat['illiquidity_21d'] = 0
                feat['hl_spread_21d'] = 0
            
            # ==========================================
            # RISK FEATURES
            # ==========================================
            
            # Sharpe-like ratio
            if len(excess_ret) >= 21 and feat['vol_21d'] > 0:
                feat['sharpe_21d'] = feat['excess_ret_21d'] / (feat['vol_21d'] * np.sqrt(21))
            else:
                feat['sharpe_21d'] = 0
            
            # Skewness and kurtosis
            if len(ret) >= 21:
                feat['skew_21d'] = stats.skew(ret[-21:])
                feat['kurt_21d'] = stats.kurtosis(ret[-21:])
            else:
                feat['skew_21d'] = 0
                feat['kurt_21d'] = 0
            
            # ==========================================
            # TAIL/JUMP FEATURES
            # ==========================================
            
            # Maximum absolute return
            if len(ret) >= 21:
                feat['max_ret_21d'] = np.max(np.abs(ret[-21:]))
                feat['min_ret_21d'] = np.min(ret[-21:])
            else:
                feat['max_ret_21d'] = 0
                feat['min_ret_21d'] = 0
            
            # Tail ratio (95th percentile / 5th percentile)
            if len(ret) >= 63:
                p95 = np.percentile(ret[-63:], 95)
                p5 = np.percentile(ret[-63:], 5)
                feat['tail_ratio'] = p95 / abs(p5) if p5 != 0 else 1
            else:
                feat['tail_ratio'] = 1
            
            features_list.append(feat)
        
        # Convert to DataFrame
        df = pd.DataFrame(features_list)
        
        # ==========================================
        # CROSS-SECTIONAL FEATURES
        # ==========================================
        
        # Market (equal-weighted)
        market_ret = returns_window.mean(axis=1)
        
        # Percentile ranks for momentum
        for col in ['ret_21d', 'ret_63d', 'excess_ret_21d']:
            if col in df.columns:
                df[f'{col}_pct'] = df[col].rank(pct=True)

        # Percentile ranks for higher order lags
        for col in [f'ret_lag_{(lag-1)*5+1}-{lag*5+1}w' for lag in range(1, 5)]:
            if col in df.columns:
                df[f'{col}_pct'] = df[col].rank(pct=True)


        for col in [f'ret_lag_{(lag-1)*21+1}-{lag*21+1}w' for lag in range(1, 3)]:
            if col in df.columns:
                df[f'{col}_pct'] = df[col].rank(pct=True)
        
        # Percentile ranks for volatility
        for col in ['vol_21d', 'vol_5d', 'vol_63d']:
            if col in df.columns:
                df[f'{col}_pct'] = df[col].rank(pct=True)
        
        # Z-scores
        for col in ['ret_21d', 'vol_21d']:
            if col in df.columns:
                mean = df[col].mean()
                std = df[col].std()
                if std > 0:
                    df[f'{col}_zscore'] = (df[col] - mean) / std
                else:
                    df[f'{col}_zscore'] = 0
        
        # Beta to market
        betas = []
        for i in range(n_stocks):
            stock_ret = returns_window.iloc[:, i].values
            if len(stock_ret) >= 60 and len(market_ret) >= 60:
                cov = np.cov(stock_ret[-60:], market_ret[-60:].values)[0, 1]
                var = np.var(market_ret[-60:].values)
                beta = cov / var if var > 0 else 1
            else:
                beta = 1
            betas.append(beta)
        
        df['beta'] = betas
        df['beta_pct'] = pd.Series(betas).rank(pct=True).values
        
        # Correlation to market
        correlations = []
        for i in range(n_stocks):
            stock_ret = returns_window.iloc[:, i].values
            if len(stock_ret) >= 60 and len(market_ret) >= 60:
                corr = np.corrcoef(stock_ret[-60:], market_ret[-60:].values)[0, 1]
            else:
                corr = 0
            correlations.append(corr)
        
        df['corr_market'] = correlations
        
        # Risk-adjusted momentum
        df['risk_adj_mom'] = df['excess_ret_21d'] / (df['vol_21d'] + 1e-8)
        df['risk_adj_mom_pct'] = df['risk_adj_mom'].rank(pct=True)
        
        # Momentum-volatility interaction
        df['mom_vol_interaction'] = df['ret_21d'] * df['vol_21d']
        
        return df


    def feature_subset_horizon(self, features: pd.DataFrame, horizon_days: int, target: str) -> pd.DataFrame:
        """
        Select feature subsets based on prediction horizon and target type
        
        Uses predefined feature pools for consistency and maintainability
        
        Args:
            features: DataFrame with all features
            horizon_days: Prediction horizon in days (1, 5, or 21)
            target: Target variable ('return' or 'volatility')
        
        Returns:
            DataFrame with selected features
        """
        
        # ==========================================
        # FEATURE POOL DEFINITIONS
        # ==========================================
        
        pools = {
            # Momentum features (different horizons)
            'momentum_short': ['ret_1d', 'ret_5d'],
            'momentum_medium': ['ret_21d', 'excess_ret_21d'],
            'momentum_long': ['ret_63d', 'excess_ret_63d'],
            
            # Volatility measures
            'volatility_short': ['vol_5d', 'rv_5d'],
            'volatility_medium': ['vol_21d', 'rv_21d', 'vol_ewma', 'downside_vol_21d'],
            'volatility_long': ['vol_63d', 'rv_63d'],
            
            # Volatility dynamics
            'vol_dynamics': ['rv_ratio_5_21', 'rv_ratio_5_63', 'rv_ratio_21_63', 'rv_trend'],
            
            # Technical indicators
            'technical_short': [f'ret_lag_{i}d' for i in range(1, 6)],  # Lags 1-5
            'technical_medium': ['price_ma20', 'momentum_accel', 'range_position', 'rsi_14'],
            'technical_long': ['price_ma50', 'drawdown_21d'],

            # Higher order lags
            'higher_order_lags_medium': [f'ret_lag_{(i-1)*5+1}-{i*5+1}w' for i in range(1, 5)],  # Weekly lags
            'higher_order_lags_high': [f'ret_lag_{(i-1)*21+1}-{i*21+1}w' for i in range(1, 3)],  # Monthly lags

            # Volume features
            'volume': ['volume_ratio', 'volume_vol_21d'],
            
            # Risk features
            'risk': ['sharpe_21d', 'skew_21d', 'kurt_21d'],
            
            # Liquidity features
            'liquidity': ['illiquidity_21d', 'hl_spread_21d'],
            
            # Tail/jump features
            'tail': ['max_ret_21d', 'min_ret_21d', 'tail_ratio'],
            
            # Cross-sectional features
            'cross_sectional_day': [
                'ret_21d_pct', 'ret_63d_pct', 'excess_ret_21d_pct',
                'vol_21d_pct', 'vol_5d_pct', 'vol_63d_pct',
                'ret_21d_zscore', 'vol_21d_zscore',
                'beta', 'beta_pct', 'corr_market',
                'risk_adj_mom', 'risk_adj_mom_pct', 'mom_vol_interaction'
            ],

            'cross_sectional_week': [
                'vol_21d_pct', 'vol_5d_pct', 'vol_63d_pct',
                'vol_21d_zscore',
                'beta', 'beta_pct', 'corr_market',
                'risk_adj_mom', 'risk_adj_mom_pct', 'mom_vol_interaction'
            ] + [f'ret_lag_{(i-1)*5+1}-{i*5+1}w_pct' for i in range(1, 5)],

            'cross_sectional_month': [
                'vol_21d_pct', 'vol_63d_pct',
                'vol_21d_zscore',
                'beta', 'beta_pct', 'corr_market',
                'risk_adj_mom', 'risk_adj_mom_pct', 'mom_vol_interaction'
            ] + [f'ret_lag_{(i-1)*21+1}-{i*21+1}w_pct' for i in range(1, 3)],
        }
        
        # ==========================================
        # SELECTION RULES BY (TARGET, HORIZON)
        # ==========================================
        
        selection_rules = {
            # === RETURN PREDICTION ===
            
            ('return', 1): {  # Daily returns
                'pools': [
                    'momentum_short',      # Use very recent returns
                    'volatility_short',    # Recent volatility
                    'volatility_medium',   # For risk adjustment
                    'technical_short',     # Lagged returns 1-5d
                    'technical_medium',    # MA, RSI, momentum accel
                    'volume',              # Volume signals
                    'risk',                # Sharpe, skew, kurtosis
                    'liquidity',           # Illiquidity, spread
                    'cross_sectional_day'      # Rankings, beta
                ],
                'exclude': [],
                'rationale': 'Daily: Use all high-frequency signals and microstructure'
            },
            
            ('return', 5): {  # Weekly returns
                'pools': [
                    'volatility_medium',   # 21d vol for risk
                    'technical_medium',    # MA, range position
                    'technical_long',      # Longer MA, drawdown
                    'higher_order_lags_medium',  # Weekly lags
                    'volume',              # Volume signals
                    'risk',                # Risk metrics
                    'liquidity',           # Liquidity matters
                    'cross_sectional_week'      # Rankings
                ],
                'exclude': ['ret_1d', 'mom_vol_interaction', 'volume_ratio', 'beta'] + [f'ret_lag_{i}d' for i in range(1, 6)],  # Drop daily lags
                'rationale': 'Weekly: Drop daily noise, keep medium-term momentum'
            },
            
            ('return', 21): {  # Monthly returns
                'pools': [
                    'volatility_medium',   # 21d vol
                    'volatility_long',     # 63d vol
                    'technical_long',      # Long-term technical
                    'higher_order_lags_high',  # Monthly lags
                    'volume',              # Volume
                    'risk',                # Risk metrics
                    'cross_sectional_month'      # Rankings
                ],
                'exclude': ['vol_5d_pct', 'rv_5d'] +
                        [f'ret_lag_{i}d' for i in range(1, 21)],  # Drop all short-term
                'rationale': 'Monthly: Focus on persistent signals, drop short-term noise'
            },
            
            # === VOLATILITY PREDICTION ===
            
            ('volatility', 5): {  # Weekly volatility
                'pools': [
                    'volatility_short',    # Recent vol
                    'volatility_medium',   # 21d vol, EWMA
                    'vol_dynamics',        # RV ratios, trends
                    'volume',              # Volume signals
                    'risk',                # Skew, kurtosis
                    'tail',                # Jump indicators
                    'liquidity',           # Liquidity affects vol
                    'cross_sectional_week'      # Rankings
                ],
                'exclude': ['ret_21_d', 'excess_ret_21d', 'ret_63d', 'excess_ret_63d', 'ret_21d_zscore'] ,  # Drop return features
                'rationale': 'Weekly vol: Use recent variance measures and tail risk'
            },
            
            ('volatility', 21): {  # Monthly volatility
                'pools': [
                    'volatility_medium',   # 21d vol
                    'volatility_long',     # 63d vol
                    'vol_dynamics',        # RV dynamics
                    'volume',              # Volume
                    'risk',                # Higher moments
                    'tail',                # Tail risk
                    'liquidity',           # Liquidity
                    'cross_sectional_month'      # Rankings
                ],
                'exclude': ['vol_5d', 'rv_5d', 'rv_ratio_5_21', 'rv_ratio_5_63'] + ['ret_21_d', 'excess_ret_21d', 'ret_63d', 'excess_ret_63d', 'ret_21d_zscore'],  # Drop short-term
                'rationale': 'Monthly vol: Focus on longer-term variance persistence'
            }
        }
        
        # ==========================================
        # FEATURE SELECTION LOGIC
        # ==========================================
        
        key = (target, horizon_days)
        if key not in selection_rules:
            raise ValueError(f"No selection rule for target={target}, horizon={horizon_days}. "
                            f"Valid combinations: {list(selection_rules.keys())}")
        
        rule = selection_rules[key]
        
        # Build feature list from pools
        selected_features = ['ticker']  # Always include ticker
        for pool_name in rule['pools']:
            if pool_name in pools:
                selected_features.extend(pools[pool_name])
        
        # Remove excluded features
        selected_features = [f for f in selected_features if f not in rule['exclude']]
        
        # Remove duplicates (maintain order)
        seen = set()
        selected_features = [f for f in selected_features if not (f in seen or seen.add(f))]
        
        # Filter to features that actually exist in DataFrame
        available_features = [f for f in selected_features if f in features.columns]
        
        # Warn about missing features
        missing = set(selected_features) - set(available_features) - {'ticker'}
        if missing:
            print(f"⚠ Warning: {len(missing)} expected features not found in data:")
            print(f"  Missing: {sorted(missing)[:10]}")  # Show first 10
        
        # Print summary
        print(f"\n✓ Selected {len(available_features)-1} features for {target} prediction (horizon={horizon_days}d)")
        print(f"  Pools used: {rule['pools']}")
        if rule['exclude']:
            print(f"  Excluded: {len(rule['exclude'])} features")
        
        return features[available_features]

In [ ]:
with open('src/processed_data.pkl', 'rb') as f:
    op, high, low, close, volume, returns, risk_free_rate = pickle.load(f)

# Create engineer
engineer = FeatureEngineer(op, high, low, close, volume, returns, risk_free_rate, lookback_days=70)

# Define functions
def func1():
    features, df = engineer.compute_monthly_volatility_features('src/data_monthly_var/')
    print(f"Monthly vol: {features.shape}, targets: {df.shape}")

def func3():
    features, df = engineer.compute_monthly_return_features('src/data_monthly_ret/')
    print(f"Monthly ret: {features.shape}, targets: {df.shape}")

def func4():
    features, df = engineer.compute_weekly_return_features('src/data_weekly_ret/')
    print(f"Weekly ret: {features.shape}, targets: {df.shape}")

def func5():
    features, df = engineer.compute_weekly_volatility_features('src/data_weekly_var/')
    print(f"Weekly vol: {features.shape}, targets: {df.shape}")

def func6():
    features, df = engineer.compute_daily_return_features('src/data_daily_ret/')
    print(f"Daily ret: {features.shape}, targets: {df.shape}")

if __name__ == '__main__':  # IMPORTANT: Protect entry point
    # Create processes
    p1 = Process(target=func1)
    p3 = Process(target=func3)
    p4 = Process(target=func4)
    p5 = Process(target=func5)
    p6 = Process(target=func6)
    
    # Start all
    p1.start()
    p3.start()
    p4.start()
    p5.start()
    p6.start()
    
    # Wait for completion
    p1.join()
    p3.join()
    p4.join()
    p5.join()
    p6.join()

    features, df = engineer.compute_daily_return_features('src/data_daily_vol/')
    
    print("✓ All feature engineering complete!")

For our test data we use 2017-02-08 to 2018-02-07

For our validation data we use 2014-03-03 to 2016-10-03

In [ ]:
with open('src/processed_data.pkl', 'rb') as f:
    op, high, low, close, volume, returns, risk_free_rate = pickle.load(f)

train_returns = returns.iloc[:-252]
test_returns = returns.iloc[-252:]
test_risk_free_rate = risk_free_rate.iloc[-252:]
cv_returns = train_returns.iloc[-252:]
cv_returns

Cross Validation to choose the models to use

In [ ]:
class MLReturnPredictor:
    """ML-based return prediction"""
    
    def __init__(self, model_type: str = 'ridge', **model_params):
        """
        Args:
            model_type: 'Ridge' or 'RF' or 'Lasso' or 'ENet' or 'XGB'
            **model_params: Model hyperparameters
        """
        self.model_type = model_type
        self.model_params = model_params
        self.model = None
    
    def create_training_dataset(self,
                               feature_files,
                               data_source,
                               returns_df: pd.DataFrame):
        """
        Create training dataset from feature files and returns
        
        Args:
            feature_files: List of feature pickle file paths
            returns_df: DataFrame (dates × stocks) with returns
        
        Returns:
            X_train: (N_samples, N_features)
            y_train: (N_samples,)
        """
        print(f"Creating training dataset from {len(feature_files)} files...")
        
        X_list = []
        y_list = []
        for feature_file in feature_files:
            # Extract date from filename
            feature_date = pd.to_datetime(feature_file.replace(".pkl", ""))
            
            # Load features
            with open(f"src/data_{data_source}/{feature_file}", 'rb') as f:
                features_df = pickle.load(f)
            
            # Get target returns (1-day ahead)
            if feature_date not in returns_df.index:
                continue
            
            target_returns = returns_df.loc[feature_date]
            
            # Align stocks
            common_stocks = features_df.index.intersection(target_returns.index)
            if len(common_stocks) == 0:
                continue
            
            # Store
            X_list.append(features_df.loc[common_stocks].values)
            y_list.append(target_returns.loc[common_stocks].values)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            
        
        X_train = np.vstack(X_list)
        y_train = np.concatenate(y_list)
        
        print(f"✓ Dataset created: X={X_train.shape}, y={y_train.shape}")
        return X_train, y_train
    
    def train(self, X_train: np.ndarray, y_train: np.ndarray):
        """
        Train the model
        
        Args:
            X_train: Training features
            y_train: Training targets
        """
        print(f"Training {self.model_type} model...")
        
        if self.model_type == 'Ridge':
            params = {'alpha': 1.0, **self.model_params}
            self.model = Ridge(**params)
        elif self.model_type == 'RF':
            params = {
                'n_estimators': 100,
                'max_depth': 10,
                'min_samples_leaf': 100,
                'n_jobs': -1,
                **self.model_params
            }
            self.model = RandomForestRegressor(**params)
        elif self.model_type == 'Lasso':
            params = {'alpha': 0.1, **self.model_params}
            self.model = Lasso(**params)
        elif self.model_type == 'ENet':
            params = {'alpha': 0.1, 'l1_ratio': 0.5, **self.model_params}
            self.model = ElasticNet(**params) 
        elif self.model_type == 'XGB':
            params = {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1, **self.model_params}
            self.model = XGBRegressor(**params)
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")
        
        self.model.fit(X_train, y_train)
        
        # Quick diagnostics
        y_pred = self.model.predict(X_train)
        r2 = r2_score(y_train, y_pred)
        
        print(f"✓ Training R²: {r2:.4f}")
        print(f"  Prediction spread: {y_pred.std():.4%}")

    def predict_all_test_returns(self,
                                 feature_files,
                                 data_source,
                                 scaler: StandardScaler = None):
        """
        Predict returns for all test dates
        
        Args:
            feature_files: List of test feature files
        
        Returns:
            predicted_returns_df: DataFrame (dates × stocks)
        """
        print(f"Predicting returns for {len(feature_files)} dates...")
        
        predictions_dict = {}
        
        for feature_file in feature_files:
            # Extract date
            date_str = Path(feature_file).stem.replace('features_', '')
            pred_date = pd.to_datetime(date_str)
            
            # Load and predict
            with open(f"src/data_{data_source}/{feature_file}", 'rb') as f:
                features_df = pickle.load(f)
            
            scaled_features = features_df.values
            if scaler is not None:
                scaled_features = scaler.transform(scaled_features)
            predictions = self.model.predict(scaled_features)
            predictions_dict[pred_date] = pd.Series(predictions, index=features_df.index)
        
        predicted_returns_df = pd.DataFrame(predictions_dict).T
        
        print(f"✓ Predictions complete: {predicted_returns_df.shape}")
        return predicted_returns_df

In [ ]:
def evaluate_predictions(predicted_df, actual_df, for_returns=True):
    """
    Evaluate prediction quality for returns
    
    Args:
        predicted_df: DataFrame (dates × stocks) with predicted returns
        actual_df: DataFrame (dates × stocks) with actual returns
    """
    
    # Align dataframes
    common_dates = predicted_df.index.intersection(actual_df.index)
    common_stocks = predicted_df.columns.intersection(actual_df.columns)
    
    pred = predicted_df.loc[common_dates, common_stocks]
    actual = actual_df.loc[common_dates, common_stocks]
    
    # Flatten to 1D arrays (drop NaNs)
    pred_flat = pred.values.flatten()
    actual_flat = actual.values.flatten()
    
    valid_mask = ~(np.isnan(pred_flat) | np.isnan(actual_flat))
    pred_flat = pred_flat[valid_mask]
    actual_flat = actual_flat[valid_mask]

    if not for_returns:
        r2 = r2_score(actual_flat, pred_flat)
        print(f"\n5. R2 SCORE: {r2:.4f}")

        rmse = np.sqrt(np.mean((pred_flat - actual_flat) ** 2))
        print(f"   RMSE: {rmse:.4f}")

        mae = np.mean(np.abs(pred_flat - actual_flat))
        print(f"   MAE: {mae:.4f}")

        return {'r2_score': r2,
                'rmse': rmse,
                'mae': mae}   
    
    print("="*60)
    print("PREDICTION EVALUATION")
    print("="*60)
    print(f"Total observations: {len(pred_flat):,}")
    print(f"Dates: {len(common_dates)}, Stocks: {len(common_stocks)}")
    
    # 1. Directional Accuracy
    direction_correct = (np.sign(pred_flat) == np.sign(actual_flat)).mean()
    print(f"\n1. DIRECTIONAL ACCURACY: {direction_correct:.2%}")
    
    if direction_correct > 0.52:
        print("    Good - better than random")
    elif direction_correct > 0.50:
        print("   ~ Weak signal")
    else:
        print("   ✗ Worse than random!")
    
    # 2. Information Coefficient (Spearman)
    ic, ic_pval = spearmanr(pred_flat, actual_flat)
    print(f"\n2. INFORMATION COEFFICIENT: {ic:.4f} (p={ic_pval:.4f})")
    
    if ic > 0.05:
        print("    Good - usable signal")
    elif ic > 0.02:
        print("   ~ Weak but non-zero")
    else:
        print("   ✗ Very weak signal")
    
    # 3. Pearson Correlation (for comparison)
    pearson_corr = np.corrcoef(pred_flat, actual_flat)[0, 1]
    print(f"\n3. PEARSON CORRELATION: {pearson_corr:.4f}")
    
    # 4. Confusion Matrix
    pred_sign = np.sign(pred_flat)
    actual_sign = np.sign(actual_flat)
    
    # Map to Up/Down (ignore zeros for clarity)
    pred_binary = (pred_sign > 0).astype(int)
    actual_binary = (actual_sign > 0).astype(int)
    
    cm = confusion_matrix(actual_binary, pred_binary)
    
    print(f"\n4. CONFUSION MATRIX:")
    print(f"                 Predicted")
    print(f"                 Down    Up")
    print(f"Actual  Down  [{cm[0,0]:6d} {cm[0,1]:6d}]")
    print(f"        Up    [{cm[1,0]:6d} {cm[1,1]:6d}]")
    
    # Precision and Recall
    precision = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1] + cm[0,1]) > 0 else 0
    recall = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1] + cm[1,0]) > 0 else 0
    
    print(f"\n   Precision (when predict Up): {precision:.2%}")
    print(f"   Recall (catch actual Ups): {recall:.2%}")

    # 5. R2 Score
    r2 = r2_score(actual_flat, pred_flat)
    print(f"\n5. R2 SCORE: {r2:.4f}")
    
    return {
        'directional_accuracy': direction_correct,
        'information_coefficient': ic,
        'pearson_correlation': pearson_corr,
        'confusion_matrix': cm,
        'precision': precision,
        'recall': recall,
        'r2_score': r2
    }

In [ ]:
with open('src/data_weekly_ret/return_5_targets.pkl', 'rb') as f:
    weekly_volatility_targets = pickle.load(f)

mask = (weekly_volatility_targets.index >= '2014-03-03') & (weekly_volatility_targets.index <= '2016-10-03')
testing_data = weekly_volatility_targets.loc[mask]
testing_dates = testing_data.index.strftime('%Y-%m-%d').map(lambda x: x+".pkl").values

results_df = {}

for method in ['Ridge', 'Lasso', 'ENet', 'RF', 'XGB']:
    model_params = []
    if method == 'Ridge':
        alphas = [1e-06, 1e-07, 1e-08, 1e-09, 1e-10] # 5 models
        model_params = [{'alpha': a} for a in alphas]
    elif method == 'Lasso':
        alphas = [.0001, 1e-05, 1e-06, 1e-07, 1e-08] # 5 models
        model_params = [{'alpha': a} for a in alphas]
    elif method == 'ENet':
        alphas = [1e-06, 1e-07, 1e-08]
        l1_ratios = [.3, .5, .7]
        model_params = [{'alpha': a, 'l1_ratio': r} for a, r in product(alphas, l1_ratios)] #9 models
    elif method == 'RF':
        n_estimators=[10, 100]
        max_depth=[3, 5]
        min_samples_leaf=[1000, 5000]
        max_features=['sqrt']
        n_jobs=-1
        model_params = [{'n_estimators': n, 'max_depth': d, 'min_samples_leaf': l, 'max_features': f, 'n_jobs': j}
                        for n, d, l, f, j in product(n_estimators, max_depth, min_samples_leaf, max_features, [n_jobs])] #8 models
    elif method == 'XGB':
        n_estimators=[10, 100]
        max_depth=[3, 5]
        learning_rate=[0.01, 0.1]
        min_child_weight=[100, 300]
        reg_lambda=[.1, 1]
        subsample=[0.8]
        colsample_bytree=[0.8]
        n_jobs=-1
        model_params = [{'n_estimators': n, 'max_depth': d, 'learning_rate': lr,
                         'min_child_weight': mcw, 'reg_lambda': rl,
                         'subsample': ss, 'colsample_bytree': cb,
                         'n_jobs': j}
                        for n, d, lr, mcw, rl, ss, cb, j in
                        product(n_estimators, max_depth, learning_rate,
                                min_child_weight, reg_lambda,
                                subsample, colsample_bytree, [n_jobs])] #16 models
    
    def model_cv(model_params) -> dict:
        preds = []

        for test_date in testing_dates:
            predictor = MLReturnPredictor(model_type=method, **model_params)
            v = os.listdir('src/data_weekly_ret')[:os.listdir('src/data_weekly_ret').index(test_date)-14]
            X_train, y_train = predictor.create_training_dataset(v, "weekly_ret", weekly_volatility_targets)
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            predictor.train(X_train, y_train)
            predicted = predictor.predict_all_test_returns([test_date], "weekly_ret", scaler)
            preds.append(predicted)

        predicted = pd.concat(preds)
        return predicted

    weights_list = Parallel(n_jobs=-1)(
        delayed(model_cv)(d) for d in tqdm(model_params)
    )

    for i in range(len(weights_list)):
        model_param = model_params[i]
        results = evaluate_predictions(weights_list[i], testing_data)
        results_df[(method, frozenset(model_param.items()))] = results
results_df = pd.DataFrame(results_df).T
results_df.to_pickle('src/cv_results/weekly_return_model_results.pkl')

In [ ]:
results_df.sort_values(('information_coefficient'), ascending=False)

In [ ]:
with open('src/data_daily_ret/return_1_targets.pkl', 'rb') as f:
    daily_volatility_targets = pickle.load(f)

mask = (daily_volatility_targets.index >= '2014-03-03') & (daily_volatility_targets.index <= '2016-10-03')
testing_data = daily_volatility_targets.loc[mask]
testing_dates = testing_data.index.strftime('%Y-%m-%d').map(lambda x: x+".pkl").values

results_df = {}

for method in ['Ridge', 'Lasso', 'ENet', 'RF']:
    model_params = []
    if method == 'Ridge':
        alphas = [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9] # 7 models
        model_params = [{'alpha': a} for a in alphas]
    elif method == 'Lasso':
        alphas = [.01, .001, .0001, 1e-05, 1e-06, 1e-07] # 6 models
        model_params = [{'alpha': a} for a in alphas]
    elif method == 'ENet':
        alphas = [.01, .001, .0001, 1e-05, 1e-06, 1e-07]
        l1_ratios = [.3, .5, .7]
        model_params = [{'alpha': a, 'l1_ratio': r} for a, r in product(alphas, l1_ratios)] #20 models
    elif method == 'RF':
        n_estimators=[10, 100]
        max_features=['sqrt']
        n_jobs=-1
        model_params = [{'n_estimators': n, 'max_depth': d, 'min_samples_leaf': l, 'max_features': f, 'n_jobs': j}
                        for n, d, l, f, j in product(n_estimators, max_depth, min_samples_leaf, max_features, [n_jobs])] #24 models
    elif method == 'XGB':
        n_estimators=[10, 100]
        min_child_weight=[100, 200]
        subsample=[0.8]
        colsample_bytree=[0.8]
        n_jobs=-1
        model_params = [{'n_estimators': n, 'max_depth': d, 'learning_rate': lr,
                         'min_child_weight': mcw, 'reg_lambda': rl,
                         'subsample': ss, 'colsample_bytree': cb,
                         'n_jobs': j}
                        for n, d, lr, mcw, rl, ss, cb, j in
                        product(n_estimators, max_depth, learning_rate,
                                min_child_weight, reg_lambda,
                                subsample, colsample_bytree, [n_jobs])]
    def model_cv(model_params) -> dict:
        preds = []

        for test_date in testing_dates:
            predictor = MLReturnPredictor(model_type=method, **model_params)
            v = os.listdir('src/data_daily_ret')[:os.listdir('src/data_daily_ret').index(test_date)-70]
            X_train, y_train = predictor.create_training_dataset(v, "daily_ret", daily_volatility_targets)
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            predictor.train(X_train, y_train*10000)  # Scale up targets for daily returns
            predicted = predictor.predict_all_test_returns([test_date], "daily_ret", scaler)/10000  # Scale down predictions
            preds.append(predicted)

        predicted = pd.concat(preds)
        return predicted

    weights_list = Parallel(n_jobs=-1)(
        delayed(model_cv)(d) for d in tqdm(model_params)
    )

    for i in range(len(weights_list)):
        model_param = model_params[i]
        results = evaluate_predictions(weights_list[i], testing_data)
        results_df[(method, frozenset(model_param.items()))] = results
results_df = pd.DataFrame(results_df).T
results_df.to_pickle('src/cv_results/daily_return_model_results.pkl')

In [ ]:
with open('src/monthly_return_model_results.pkl', 'rb') as f:
    results_df = pickle.load(f)
results_df.sort_values(('information_coefficient'), ascending=False).head(10)

In [ ]:
from itertools import product
import random
random.seed(4212)

with open('src/data_monthly_ret/return_21_targets.pkl', 'rb') as f:
    monthly_volatility_targets = pickle.load(f)

mask = (monthly_volatility_targets.index >= '2014-03-03') & (monthly_volatility_targets.index <= '2016-10-03')
testing_data = monthly_volatility_targets.loc[mask]
testing_dates = testing_data.index.strftime('%Y-%m-%d').map(lambda x: x+".pkl").values

results_df = {}

for method in ['Ridge', 'Lasso', 'ENet', 'RF', 'XGB']:
    model_params = []
    if method == 'Ridge':
        alphas = [1e-05, 1e-06, 1e-07, 1e-08, 1e-09, 1e-10] # 6 models
        model_params = [{'alpha': a} for a in alphas]
    elif method == 'Lasso':
        alphas = [.01, .001, .0001, 1e-05, 1e-06, 1e-07] # 6 models
        model_params = [{'alpha': a} for a in alphas]
    elif method == 'ENet':
        alphas = [.01, .001, .0001]
        l1_ratios = [.1, .3, .5, .7, .9]
        model_params = [{'alpha': a, 'l1_ratio': r} for a, r in product(alphas, l1_ratios)] #20 models
    elif method == 'RF':
        n_estimators=[10, 100, 500]
        max_depth=[3, 5, 7]
        min_samples_leaf=[50, 100, 500]
        max_features=['sqrt']
        n_jobs=-1
        model_params = [{'n_estimators': n, 'max_depth': d, 'min_samples_leaf': l, 'max_features': f, 'n_jobs': j}
                        for n, d, l, f, j in product(n_estimators, max_depth, min_samples_leaf, max_features, [n_jobs])] #24 models
    elif method == 'XGB':
        n_estimators=[10, 100, 200]
        max_depth=[3, 5, 7]
        learning_rate=[0.05, 0.1]
        min_child_weight=[100, 200]
        reg_lambda=[.1, 1, 5]
        subsample=[0.8]
        colsample_bytree=[0.8]
        n_jobs=-1
        model_params = [{'n_estimators': n, 'max_depth': d, 'learning_rate': lr,
                         'min_child_weight': mcw, 'reg_lambda': rl,
                         'subsample': ss, 'colsample_bytree': cb,
                         'n_jobs': j}
                        for n, d, lr, mcw, rl, ss, cb, j in
                        product(n_estimators, max_depth, learning_rate,
                                min_child_weight, reg_lambda,
                                subsample, colsample_bytree, [n_jobs])]
    def model_cv(model_params) -> dict:
        preds = []

        for test_date in testing_dates:
            predictor = MLReturnPredictor(model_type=method, **model_params)
            v = os.listdir('src/data_monthly_ret')[:os.listdir('src/data_monthly_ret').index(test_date)-4]
            X_train, y_train = predictor.create_training_dataset(v, "monthly_ret", monthly_volatility_targets)
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            predictor.train(X_train, y_train)
            predicted = predictor.predict_all_test_returns([test_date], "monthly_ret", scaler)
            preds.append(predicted)

        predicted = pd.concat(preds)
        return predicted

    weights_list = Parallel(n_jobs=-1)(
        delayed(model_cv)(d) for d in tqdm(model_params)
    )

    for i in range(len(weights_list)):
        model_param = model_params[i]
        results = evaluate_predictions(weights_list[i], testing_data)
        results_df[(method, frozenset(model_param.items()))] = results
results_df = pd.DataFrame(results_df).T
results_df.to_pickle('src/cv_results/monthly_return_model_results.pkl')

In [ ]:
with open('src/cv_results/monthly_volatility_model_results.pkl', 'rb') as f:
    results_df = pickle.load(f)
results_df.sort_values(('r2_score'), ascending=False).head(10)

In [ ]:
from itertools import product
import random
random.seed(4212)

with open('src/data_monthly_var/volatility_21_targets.pkl', 'rb') as f:
    monthly_volatility_targets = pickle.load(f)

mask = (monthly_volatility_targets.index >= '2014-03-03') & (monthly_volatility_targets.index <= '2016-10-03')
testing_data = monthly_volatility_targets.loc[mask]
testing_dates = testing_data.index.strftime('%Y-%m-%d').map(lambda x: x+".pkl").values

results_df = {}

for method in ['Ridge', 'Lasso', 'ENet', 'RF', 'XGB']:
    model_params = []
    if method == 'Ridge':
        alphas = [1e-08, 1e-09, 1e-10, 1e-11, 1e-12] # 5 models
        model_params = [{'alpha': a} for a in alphas]
    elif method == 'Lasso':
        alphas = [.001, .0001, 1e-05, 1e-06] # 4 models
        model_params = [{'alpha': a} for a in alphas]
    elif method == 'ENet':
        alphas = [.001, .0001, 1e-05, 1e-06]
        l1_ratios = [.3, .5, .7]
        model_params = [{'alpha': a, 'l1_ratio': r} for a, r in product(alphas, l1_ratios)] #20 models
    elif method == 'RF':
        n_estimators=[10, 100, 500]
        max_depth=[7, 9]
        min_samples_leaf=[25, 50, 100]
        max_features=['sqrt']
        n_jobs=-1
        model_params = [{'n_estimators': n, 'max_depth': d, 'min_samples_leaf': l, 'max_features': f, 'n_jobs': j}
                        for n, d, l, f, j in product(n_estimators, max_depth, min_samples_leaf, max_features, [n_jobs])] #24 models
    elif method == 'XGB':
        n_estimators=[100, 500]
        max_depth=[5, 7, 9]
        learning_rate=[0.01, 0.1, 1]
        min_child_weight=[100, 500]
        reg_lambda=[.1, 1, 10]
        subsample=[0.8]
        colsample_bytree=[0.8]
        n_jobs=-1
        model_params = [{'n_estimators': n, 'max_depth': d, 'learning_rate': lr,
                         'min_child_weight': mcw, 'reg_lambda': rl,
                         'subsample': ss, 'colsample_bytree': cb,
                         'n_jobs': j}
                        for n, d, lr, mcw, rl, ss, cb, j in
                        product(n_estimators, max_depth, learning_rate,
                                min_child_weight, reg_lambda,
                                subsample, colsample_bytree, [n_jobs])]
    def model_cv(model_params) -> dict:
        preds = []

        for test_date in testing_dates:
            predictor = MLReturnPredictor(model_type=method, **model_params)
            v = os.listdir('src/data_monthly_var')[:os.listdir('src/data_monthly_var').index(test_date)-4]
            X_train, y_train = predictor.create_training_dataset(v, "monthly_var", monthly_volatility_targets)
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            predictor.train(X_train, np.log(y_train))
            predicted = np.exp(predictor.predict_all_test_returns([test_date], "monthly_var", scaler))
            preds.append(predicted)

        predicted = pd.concat(preds)
        return predicted

    weights_list = Parallel(n_jobs=-1)(
        delayed(model_cv)(d) for d in tqdm(model_params)
    )

    for i in range(len(weights_list)):
        model_param = model_params[i]
        results = evaluate_predictions(weights_list[i], testing_data, for_returns=False)
        results_df[(method, frozenset(model_param.items()))] = results
results_df = pd.DataFrame(results_df).T
results_df.to_pickle('src/cv_results/monthly_volatility_model_results.pkl')

In [ ]:
p = {}

for date in testing_data.index:
    working = returns.loc[:date].iloc[:-1]
    working = working.mean()
    p[date] = working
p = pd.DataFrame(p).T
p.index = testing_data.index
p.set_index(testing_data.index, inplace=True)
evaluate_predictions(p, testing_data)

In [ ]:
features = []
for i in range(X_train.T.shape[0]):
    features.append(np.corrcoef(X_train.T[i], y_train)[0, 1])

with open('src/data_weekly_ret/2013-06-10.pkl', 'rb') as f:
    f = pickle.load(f)

feature_importances = pd.Series(features, index=[f'Feature_{i}' for i in range(X_train.T.shape[0])])
feature_importances.index = f.columns
feature_importances = feature_importances.abs().sort_values(ascending=False)
print("Top 10 important features:")
print(feature_importances.head(10))
feature_importances


Purge period either 70 trading days, 14 trasing weeks, 4 trading months

Monthly volatility
Best by r2 reg_lambda:1, n_estimator:500, max_depth:9, min_child_weight:100, learning rate:.1 r2 .055584, rmse .193038, mae.043199, optimising for extreme values
rmse same as r2
Best by MAE which may be more impt as there are outliers, optimising for everyday performance,
rf, min samples leaf 25, n_estimators:500, max_depth:9

In [ ]:
with open('src/cv_results/monthly_volatility_model_results.pkl', 'rb') as f:
    results_df = pickle.load(f)
results_df.sort_values(('mae'), ascending=True).head(10)

In [ ]:
# testing_dates = test_returns.index.strftime('%Y-%m-%d').map(lambda x: x+".pkl").values

with open('src/data_monthly_var/volatility_21_targets.pkl', 'rb') as f:
    monthly_volatility_targets = pickle.load(f)

testing_data = monthly_volatility_targets.loc["2017-02-08":]
testing_dates = testing_data.index.strftime('%Y-%m-%d').map(lambda x: x+".pkl").values

preds = []

model_params = {'n_estimators': 500,
                'max_depth': 9,
                'learning_rate': 0.1,
                'min_child_weight': 100,
                'reg_lambda': 1,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'n_jobs': -1}

for test_date in testing_dates:
    predictor = MLReturnPredictor(model_type='XGB', **model_params)
    v = os.listdir('src/data_monthly_var')[:os.listdir('src/data_monthly_var').index(test_date)-4]
    X_train, y_train = predictor.create_training_dataset(v, "monthly_var", monthly_volatility_targets)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    predictor.train(X_train, np.log(y_train))
    predicted = np.exp(predictor.predict_all_test_returns([test_date], "monthly_var", scaler))
    preds.append(predicted)

predicted = pd.concat(preds)
predicted.to_pickle('src/test_predicted/monthly_volatility_predicted.pkl')

Generate weights for variance

In [ ]:
with open('src/test_predicted/monthly_volatility_predicted.pkl', 'rb') as f:
    predicted = pickle.load(f)

tickers = predicted.columns.tolist()

def compute_weights_for_date(date):
    opt = PortfolioOptimizer(returns.loc[:date].iloc[:-1]).minimum_variance_from_cov(predicted.loc[[date]], max_position=0.05)
    return pd.Series(opt, index=tickers)

weights_list = Parallel(n_jobs=-1)(
    delayed(compute_weights_for_date)(d) for d in tqdm(predicted.index, desc="weights")
)

weights = pd.concat(weights_list, axis=1).T
weights.index.name = 'date'
weights.index = predicted.index
weights.to_pickle('src/weights_predicted/monthly_volatility_weights.pkl')

The one below takes very long btw

Chooses optimal portfolio based on predicted returns

In [ ]:
tickers = predicted.columns.tolist()

def compute_weights_for_date(date):
    predicted_returns = pd.concat([returns.loc[:date].iloc[:-1], predicted.loc[[date]]])
    opt = PortfolioOptimizer(predicted_returns).minimum_variance(max_position=0.05)
    return pd.Series(opt, index=tickers)

weights_list = Parallel(n_jobs=-1)(
    delayed(compute_weights_for_date)(d) for d in tqdm(predicted.index[:5], desc="weights")
)

weights = pd.concat(weights_list, axis=1).T
weights.index.name = 'date'
weights.index = predicted.index[:5]

with open('weights.pkl', 'wb') as f:
    pickle.dump(weights, f)


Preparing backtest


In [ ]:
backtester = WalkForwardBacktest(
    train_returns=train_returns,
    test_returns=test_returns,
    # test_prices=test_prices,
    rf_rate_test=test_risk_free_rate,
    rebalance_freq='D'  # Weekly
)

running backtest

In [ ]:
results = {}

with open('weights.pkl', 'rb') as f:
    weights = pickle.load(f)

# Run Equal Weight
results['ML w fees'] = backtester.run_backtest_preloaded_weights(
    strategy_name='ML',
    preloaded_weights=weights,
    transaction_cost_bps=10
)

results['ML no fees'] = backtester.run_backtest_preloaded_weights(
    strategy_name='ML',
    preloaded_weights=weights,
    transaction_cost_bps=0
)


weights = weights.iloc[[0]]

for col in weights.columns:
    weights[col].values[:] = 1/470

results['Base - Buy & Hold'] = backtester.run_backtest_preloaded_weights(
    strategy_name='ML',
    preloaded_weights=weights,
    transaction_cost_bps=0
)

with open('src/weights_predicted/monthly_volatility_weights.pkl', 'rb') as f:
    weights_var = pickle.load(f)

weights_var.loc[pd.Timestamp("2017-02-08")] = weights_var.iloc[0]
weights_var.sort_index(inplace=True)
for i in range(len(weights_var.columns)):
    weights_var.iloc[0, i] = 1/470

results['monthly_vol'] = backtester.run_backtest_preloaded_weights(
    strategy_name='Monthly Volatility',
    preloaded_weights=weights_var,
    transaction_cost_bps=5
)

Evaluation

In [ ]:
results['monthly_vol']['weights_history']

In [ ]:
evaluator = PerformanceEvaluator(test_risk_free_rate)

comparison_df = evaluator.compare_strategies(results)
print("\nPerformance Comparison:")
print(comparison_df.to_string())

# ============================================================
# STEP 5: Visualize Results
# ============================================================
print("\n5. Generating Visualizations")
print("-" * 60)

fig = evaluator.plot_results(results, figsize=(15, 10))
plt.show()

fred = Fred(api_key = os.getenv("API_KEY"))
spy = fred.get_series_latest_release('SP500').loc["2017-02-08":"2018-02-07"]
spy.plot()
plt.show()
